# frd232-06 Robust Ensemble Time-CV

Version 6 upgrade focused on leaderboard robustness with stricter validation, leakage-safe historical features, and a 3-family ensemble (XGBoost + CatBoost + LightGBM).


In [ ]:
import os
import sys
import time
import random
import importlib.util
import subprocess
import warnings
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f"{pkg} is already installed")
        return
    run_cmd([sys.executable, "-m", "pip", "install", pkg])


ensure_package("xgboost")
ensure_package("catboost")
ensure_package("lightgbm")
ensure_package("optuna")
ensure_package("dotenv", "python-dotenv")
ensure_package("kaggle")

from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

from sklearn.base import clone
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, GroupKFold

from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb

pd.set_option("display.max_columns", 300)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
# -----------------------------
# Runtime / notebook config
# -----------------------------
COMPETITION = "fraud-detection-cpe-232-data-models"
NOTEBOOK_SLUG = "frd232-06-robust-ensemble-timecv"

RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True
USE_GPU_IF_AVAILABLE = True

ID_COL = "id"
LABEL_COL = "is_fraud"
TIME_COL = "time_ind"

CAT_COLS = ["transac_type", "src_acc", "dst_acc"]
BASE_NUM_COLS = [
    "amount",
    "src_bal",
    "src_new_bal",
    "dst_bal",
    "dst_new_bal",
    "is_flagged_fraud",
]

N_SPLITS = 5
RANDOM_STATE = 42

CV_OLD_STRATIFIED = True
CV_TIME_BASED = True
CV_GROUPED = True
LATEST_TIME_HOLDOUT = True
PRIMARY_CV_MODE = "time"

ENABLE_XGB = True
ENABLE_CATBOOST = True
ENABLE_LIGHTGBM = True

OPTUNA_TRIALS_PER_MODEL = 24  # balanced budget: ~72 trials total for 3 families
OPTUNA_TIMEOUT_SEC = 50 * 60
OPTUNA_TUNE_FOLDS = 3

THRESHOLD_GRID = np.linspace(0.01, 0.99, 199)
WEIGHT_GRID_STEP = 0.05

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def detect_xgb_device(use_gpu_if_available=True):
    if not use_gpu_if_available:
        return {"tree_method": "hist"}, "cpu"
    X_probe = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
    y_probe = np.array([0, 1, 0, 1], dtype=np.int32)
    try:
        probe = XGBClassifier(
            objective="binary:logistic",
            n_estimators=2,
            tree_method="hist",
            device="cuda",
            eval_metric="logloss",
            n_jobs=1,
        )
        probe.fit(X_probe, y_probe, verbose=False)
        return {"tree_method": "hist", "device": "cuda"}, "gpu"
    except Exception:
        try:
            probe = XGBClassifier(
                objective="binary:logistic",
                n_estimators=2,
                tree_method="gpu_hist",
                predictor="gpu_predictor",
                eval_metric="logloss",
                n_jobs=1,
            )
            probe.fit(X_probe, y_probe, verbose=False)
            return {"tree_method": "gpu_hist", "predictor": "gpu_predictor"}, "gpu"
        except Exception:
            return {"tree_method": "hist"}, "cpu"


def detect_catboost_task(use_gpu_if_available=True):
    if not use_gpu_if_available:
        return {"task_type": "CPU"}, "cpu"
    try:
        probe = CatBoostClassifier(
            iterations=2, verbose=False, task_type="GPU", random_seed=RANDOM_STATE
        )
        Xp = pd.DataFrame({"a": [0.0, 1.0, 2.0, 3.0]})
        yp = np.array([0, 1, 0, 1])
        probe.fit(Xp, yp)
        return {"task_type": "GPU"}, "gpu"
    except Exception:
        return {"task_type": "CPU"}, "cpu"


def detect_lgb_device(use_gpu_if_available=True):
    if not use_gpu_if_available:
        return {"device_type": "cpu"}, "cpu"
    return {"device_type": "cpu"}, "cpu"


XGB_DEVICE_PARAMS, XGB_DEVICE_MODE = detect_xgb_device(USE_GPU_IF_AVAILABLE)
CAT_DEVICE_PARAMS, CAT_DEVICE_MODE = detect_catboost_task(USE_GPU_IF_AVAILABLE)
LGB_DEVICE_PARAMS, LGB_DEVICE_MODE = detect_lgb_device(USE_GPU_IF_AVAILABLE)

print("XGB device mode:", XGB_DEVICE_MODE, XGB_DEVICE_PARAMS)
print("CatBoost mode:", CAT_DEVICE_MODE, CAT_DEVICE_PARAMS)
print("LGB mode:", LGB_DEVICE_MODE, LGB_DEVICE_PARAMS)


In [ ]:
# -----------------------------
# Data access and paths
# -----------------------------
def resolve_data_paths_fallback():
    kaggle_candidates = [
        Path("/kaggle/input/fraud-detection-cpe-232-data-models"),
        Path("/kaggle/input/fraudulent-transaction-detect"),
        Path("/kaggle/input"),
    ]
    local_candidates = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "kaggle/01-fraudulent-transaction-detect",
    ]

    candidate_pairs = []
    for base in kaggle_candidates + local_candidates:
        candidate_pairs.extend(
            [
                (base / "train.csv", base / "test.csv", base / "sample_submission.csv"),
                (
                    base / "data" / "train.csv",
                    base / "data" / "test.csv",
                    base / "data" / "sample_submission.csv",
                ),
                (
                    base / "fraud-detection-cpe-232-data-models" / "train.csv",
                    base / "fraud-detection-cpe-232-data-models" / "test.csv",
                    base
                    / "fraud-detection-cpe-232-data-models"
                    / "sample_submission.csv",
                ),
            ]
        )

    for train_path, test_path, sample_path in candidate_pairs:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path

    raise FileNotFoundError(
        "Could not find train/test/sample CSV files in Kaggle input or local folders"
    )


def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition}.zip"
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print("Competition zip download complete")
        except Exception as download_error:
            print(
                "Bulk download failed, fallback to per-file download:", download_error
            )
            for f in files_resp.files:
                file_path = data_dir / f.name
                if file_path.exists() and not force_download:
                    continue
                api_client.competition_download_file(
                    competition=competition,
                    file_name=f.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        if extract_dir.exists() and force_download:
            pass
        if not extract_dir.exists() or not any(extract_dir.iterdir()):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)

    train_path = extract_dir / "train.csv"
    test_path = extract_dir / "test.csv"
    sample_path = extract_dir / "sample_submission.csv"

    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        fallback_train = data_dir / "train.csv"
        fallback_test = data_dir / "test.csv"
        fallback_sample = data_dir / "sample_submission.csv"
        if (
            fallback_train.exists()
            and fallback_test.exists()
            and fallback_sample.exists()
        ):
            train_path, test_path, sample_path = (
                fallback_train,
                fallback_test,
                fallback_sample,
            )
        else:
            raise FileNotFoundError(
                "Could not resolve competition train/test/sample files"
            )

    return train_path, test_path, sample_path, zip_path, extract_dir


load_dotenv(".env", override=True)
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_error:
    print("Kaggle API auth not ready:", auth_error)
    print("Falling back to existing /kaggle/input or local data if available")

if RUN_DOWNLOAD:
    if api is None:
        raise RuntimeError("RUN_DOWNLOAD=True but Kaggle API auth is not available")
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / "data"
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = (
        prepare_competition_data(
            api_client=api,
            competition=COMPETITION,
            data_dir=DATA_DIR,
            force_download=FORCE_DOWNLOAD,
        )
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    BASE_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else BASE_DIR
OUTPUT_PATH = OUTPUT_DIR / f"submission_{NOTEBOOK_SLUG}.csv"
DIAGNOSTICS_PATH = OUTPUT_DIR / f"diagnostics_{NOTEBOOK_SLUG}.csv"
FOLD_TABLE_PATH = OUTPUT_DIR / f"fold_scores_{NOTEBOOK_SLUG}.csv"
ENSEMBLE_TABLE_PATH = OUTPUT_DIR / f"ensemble_table_{NOTEBOOK_SLUG}.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / f"feature_importance_{NOTEBOOK_SLUG}.csv"
CV_COMPARISON_PATH = OUTPUT_DIR / f"cv_comparison_{NOTEBOOK_SLUG}.csv"

config_df = pd.DataFrame(
    [
        {
            "competition": COMPETITION,
            "notebook_slug": NOTEBOOK_SLUG,
            "xgb_device_mode": XGB_DEVICE_MODE,
            "catboost_mode": CAT_DEVICE_MODE,
            "lgb_mode": LGB_DEVICE_MODE,
            "seed": RANDOM_STATE,
            "n_splits": N_SPLITS,
            "run_download": RUN_DOWNLOAD,
            "force_download": FORCE_DOWNLOAD,
            "run_submission": RUN_SUBMISSION,
            "primary_cv_mode": PRIMARY_CV_MODE,
            "optuna_trials_per_model": OPTUNA_TRIALS_PER_MODEL,
            "output_file": str(OUTPUT_PATH),
        }
    ]
)
display(config_df)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample path:", SAMPLE_PATH)


In [ ]:
# -----------------------------
# Load data and schema checks
# -----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

expected_train_cols = {
    "id",
    "time_ind",
    "transac_type",
    "amount",
    "src_acc",
    "src_bal",
    "src_new_bal",
    "dst_acc",
    "dst_bal",
    "dst_new_bal",
    "is_fraud",
    "is_flagged_fraud",
}
expected_test_cols = expected_train_cols - {LABEL_COL}

assert set(train_df.columns) == expected_train_cols, "Train schema mismatch"
assert set(test_df.columns) == expected_test_cols, "Test schema mismatch"
assert sample_submission.columns.tolist() == [ID_COL, LABEL_COL], (
    "Sample submission schema mismatch"
)

class_counts = train_df[LABEL_COL].value_counts().sort_index()
fraud_rate = float(train_df[LABEL_COL].mean())
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("Class counts (0, 1):")
print(class_counts.to_string())
print(f"Fraud rate: {fraud_rate:.8f} ({fraud_rate * 100:.4f}%)")


In [ ]:
# -----------------------------
# Leakage-safe historical features (past-only, fast path)
# -----------------------------
WINDOWS = [1, 6, 24, 168]
EPS = 1e-6


def _window_prior_stats_for_group(group_df, prefix):
    idx = group_df.index.to_numpy()
    times = group_df[TIME_COL].to_numpy(dtype=np.int64)
    amounts = group_df["amount"].to_numpy(dtype=np.float64)

    n = len(group_df)
    pos = np.arange(n, dtype=np.int64)
    out = {
        f"{prefix}_time_since_prev_tx": np.full(n, -1.0, dtype=np.float64),
    }

    if n > 1:
        out[f"{prefix}_time_since_prev_tx"][1:] = np.diff(times).astype(np.float64)

    prefix_sum = np.empty(n + 1, dtype=np.float64)
    prefix_sum[0] = 0.0
    prefix_sum[1:] = np.cumsum(amounts)

    for w in WINDOWS:
        left = np.searchsorted(times, times - w, side="left")
        cnt = (pos - left).astype(np.float64)
        sum_amt = prefix_sum[pos] - prefix_sum[left]
        mean_amt = np.divide(
            sum_amt, cnt, out=np.zeros(n, dtype=np.float64), where=cnt > 0
        )

        out[f"{prefix}_cnt_{w}h"] = cnt
        out[f"{prefix}_sum_amt_{w}h"] = sum_amt
        out[f"{prefix}_mean_amt_{w}h"] = mean_amt

    out[f"{prefix}_amt_div_mean_24h"] = amounts / (out[f"{prefix}_mean_amt_24h"] + EPS)
    return pd.DataFrame(out, index=idx)


def build_entity_history_features(df, entity_col, prefix):
    parts = []
    for _, g in df.groupby(entity_col, sort=False):
        parts.append(_window_prior_stats_for_group(g, prefix))
    return pd.concat(parts, axis=0).sort_index()


def build_pair_history_features(df):
    pair_key = df["src_acc"].astype(str) + "|" + df["dst_acc"].astype(str)
    g = df.groupby(pair_key, sort=False)

    prior_cnt = g.cumcount().astype(np.float64)
    csum = g["amount"].cumsum().astype(np.float64)
    prior_sum = (csum - df["amount"].to_numpy(dtype=np.float64)).astype(np.float64)
    prior_mean = np.divide(
        prior_sum,
        prior_cnt,
        out=np.zeros(len(df), dtype=np.float64),
        where=prior_cnt > 0,
    )

    prior_max = g["amount"].cummax().groupby(pair_key, sort=False).shift(1)
    prior_max = prior_max.fillna(0.0).to_numpy(dtype=np.float64)

    time_since_prev = (
        df.groupby(pair_key, sort=False)[TIME_COL]
        .diff()
        .fillna(-1.0)
        .to_numpy(dtype=np.float64)
    )

    out = pd.DataFrame(
        {
            "pair_prior_cnt": prior_cnt.to_numpy(dtype=np.float64),
            "pair_prior_sum_amt": prior_sum,
            "pair_prior_mean_amt": prior_mean,
            "pair_prior_max_amt": prior_max,
            "pair_time_since_prev_tx": time_since_prev,
        },
        index=df.index,
    )
    return out


def build_full_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    train_df["__is_train__"] = 1
    test_df["__is_train__"] = 0
    test_df[LABEL_COL] = np.nan

    full = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    full["_orig_order"] = np.arange(len(full))
    full = full.sort_values([TIME_COL, ID_COL]).reset_index(drop=True)

    for c in CAT_COLS:
        full[c] = full[c].astype(str)

    # Lightweight static engineered features.
    full["src_bal_change"] = full["src_new_bal"] - full["src_bal"]
    full["dst_bal_change"] = full["dst_new_bal"] - full["dst_bal"]
    full["amount_to_src_bal"] = full["amount"] / (full["src_bal"].abs() + 1.0)
    full["amount_to_dst_bal"] = full["amount"] / (full["dst_bal"].abs() + 1.0)

    t_src = time.time()
    print("Building src history features...")
    src_hist = build_entity_history_features(full, "src_acc", "src")
    print("Built src history features in", round(time.time() - t_src, 1), "seconds")

    t_dst = time.time()
    print("Building dst history features...")
    dst_hist = build_entity_history_features(full, "dst_acc", "dst")
    print("Built dst history features in", round(time.time() - t_dst, 1), "seconds")

    t_pair = time.time()
    print("Building pair history features...")
    pair_hist = build_pair_history_features(full)
    print("Built pair history features in", round(time.time() - t_pair, 1), "seconds")

    full = pd.concat([full, src_hist, dst_hist, pair_hist], axis=1)

    # Safety: no inf/nan in numeric engineered cols
    num_cols = full.select_dtypes(include=[np.number]).columns.tolist()
    full[num_cols] = full[num_cols].replace([np.inf, -np.inf], np.nan)
    full[num_cols] = full[num_cols].fillna(0.0)

    train_feat = full[full["__is_train__"] == 1].copy()
    test_feat = full[full["__is_train__"] == 0].copy()

    train_feat = train_feat.sort_values("_orig_order").reset_index(drop=True)
    test_feat = test_feat.sort_values("_orig_order").reset_index(drop=True)

    y = train_feat[LABEL_COL].astype(int).to_numpy()

    drop_cols = [LABEL_COL, "__is_train__", "_orig_order"]
    X_train = train_feat.drop(columns=drop_cols)
    X_test = test_feat.drop(columns=drop_cols)

    # leakage sanity checks: first tx per entity/pair should have no prior history.
    first_src = X_train.groupby("src_acc", sort=False)["src_cnt_1h"].first()
    first_dst = X_train.groupby("dst_acc", sort=False)["dst_cnt_1h"].first()
    pair_key = X_train["src_acc"].astype(str) + "|" + X_train["dst_acc"].astype(str)
    first_pair = (
        X_train.assign(_pair_key=pair_key)
        .groupby("_pair_key", sort=False)["pair_prior_cnt"]
        .first()
    )

    assert (first_src.values == 0).all(), (
        "Leakage check failed: src first-row history is not zero"
    )
    assert (first_dst.values == 0).all(), (
        "Leakage check failed: dst first-row history is not zero"
    )
    assert (first_pair.values == 0).all(), (
        "Leakage check failed: pair first-row history is not zero"
    )

    return X_train, y, X_test


start_feat = time.time()
X_train_all, y, X_test_all = build_full_features(train_df, test_df)
print("Feature build done in", round(time.time() - start_feat, 1), "seconds")
print("X_train shape:", X_train_all.shape)
print("X_test shape:", X_test_all.shape)


In [ ]:
# -----------------------------
# Metrics, thresholds, and split builders
# -----------------------------
def metric_at_threshold(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "fraud_f1": float(f1_score(y_true, pred, average="binary", zero_division=0)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "pred_rate": float(pred.mean()),
    }


def threshold_search(y_true, prob, thresholds):
    rows = []
    for t in thresholds:
        m = metric_at_threshold(y_true, prob, float(t))
        rows.append({"threshold": float(t), **m})
    df = pd.DataFrame(rows)
    best = (
        df.sort_values(
            ["macro_f1", "fraud_f1", "threshold"], ascending=[False, False, True]
        )
        .iloc[0]
        .to_dict()
    )
    return df, best


def build_stratified_splits(y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    idx = np.arange(len(y))
    return list(skf.split(idx, y))


def build_time_splits(time_series, n_splits=5):
    times = np.asarray(time_series)
    uniq = np.array(sorted(np.unique(times)))
    blocks = np.array_split(uniq, n_splits + 1)

    splits = []
    idx_all = np.arange(len(times))
    for i in range(1, len(blocks)):
        val_times = set(blocks[i].tolist())
        train_times = set(np.concatenate(blocks[:i]).tolist())

        va = idx_all[np.isin(times, list(val_times))]
        tr = idx_all[np.isin(times, list(train_times))]

        if len(tr) == 0 or len(va) == 0:
            continue
        splits.append((tr, va))

    return splits[:n_splits]


def build_group_splits(y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    idx = np.arange(len(y))
    return list(gkf.split(idx, y, groups=groups))


def build_latest_holdout_split(time_series, holdout_frac=0.12):
    times = np.asarray(time_series)
    uniq = np.array(sorted(np.unique(times)))
    cut = max(1, int(round(len(uniq) * holdout_frac)))
    holdout_times = set(uniq[-cut:].tolist())

    idx_all = np.arange(len(times))
    va = idx_all[np.isin(times, list(holdout_times))]
    tr = idx_all[~np.isin(times, list(holdout_times))]

    if len(va) == 0 or len(tr) == 0:
        raise ValueError("Invalid latest-time holdout split")

    return tr, va, min(holdout_times), max(holdout_times)


def split_integrity_checks(splits, n_rows):
    for i, (tr, va) in enumerate(splits, 1):
        assert len(set(tr).intersection(set(va))) == 0, f"Overlap in split {i}"
        assert len(tr) > 0 and len(va) > 0, f"Empty side in split {i}"
        assert int(np.max(tr)) < n_rows and int(np.max(va)) < n_rows


groups = X_train_all["src_acc"].astype(str).to_numpy()
time_series = X_train_all[TIME_COL].to_numpy()

splits_old = build_stratified_splits(y, n_splits=N_SPLITS, random_state=RANDOM_STATE)
splits_time = build_time_splits(time_series, n_splits=N_SPLITS)
splits_group = build_group_splits(y, groups=groups, n_splits=N_SPLITS)

split_integrity_checks(splits_old, len(y))
split_integrity_checks(splits_time, len(y))
split_integrity_checks(splits_group, len(y))

latest_tr, latest_va, holdout_t_min, holdout_t_max = build_latest_holdout_split(
    time_series, holdout_frac=0.12
)
assert np.max(time_series[latest_tr]) <= np.min(time_series[latest_va]), (
    "Latest holdout is not time-monotonic"
)

print("Old stratified folds:", len(splits_old))
print("Time-based folds:", len(splits_time))
print("Group folds:", len(splits_group))
print("Latest-time holdout range:", holdout_t_min, "to", holdout_t_max)


In [ ]:
# -----------------------------
# Model utilities + Optuna tuning
# -----------------------------
NUMERIC_FEATURES = [c for c in X_train_all.columns if c not in CAT_COLS]
X_train_num = X_train_all[NUMERIC_FEATURES].copy()
X_test_num = X_test_all[NUMERIC_FEATURES].copy()


def get_scale_pos_weight(y_vec):
    pos = max(1, int((y_vec == 1).sum()))
    neg = max(1, int((y_vec == 0).sum()))
    return float(neg / pos)


GLOBAL_SPW = get_scale_pos_weight(y)
print("scale_pos_weight baseline:", GLOBAL_SPW)


BASE_XGB = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 1600,
    "learning_rate": 0.04,
    "max_depth": 7,
    "min_child_weight": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.5,
    "gamma": 0.4,
    "max_delta_step": 1,
    "scale_pos_weight": GLOBAL_SPW,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "early_stopping_rounds": 120,
}

BASE_CAT = {
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": 2500,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 5.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 180,
    "auto_class_weights": "Balanced",
}
BASE_CAT.update(CAT_DEVICE_PARAMS)

BASE_LGB = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "n_estimators": 2200,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.85,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.2,
    "reg_lambda": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
    "scale_pos_weight": GLOBAL_SPW,
}
BASE_LGB.update(LGB_DEVICE_PARAMS)


def select_splits_for_tuning():
    if PRIMARY_CV_MODE == "time":
        return splits_time[:OPTUNA_TUNE_FOLDS]
    if PRIMARY_CV_MODE == "group":
        return splits_group[:OPTUNA_TUNE_FOLDS]
    return splits_old[:OPTUNA_TUNE_FOLDS]


TUNE_SPLITS = select_splits_for_tuning()


def optuna_objective_xgb(trial):
    params = BASE_XGB.copy()
    params.update(
        {
            "n_estimators": trial.suggest_int("n_estimators", 900, 2200),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.08, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 4, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 2, 16),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 8.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 4.0),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", GLOBAL_SPW * 0.6, GLOBAL_SPW * 1.6
            ),
        }
    )
    params.update(XGB_DEVICE_PARAMS)

    oof = np.zeros(len(y), dtype=np.float64)
    used = np.zeros(len(y), dtype=bool)

    for fold, (tr, va) in enumerate(TUNE_SPLITS, 1):
        model = XGBClassifier(**params)
        model.fit(
            X_train_num.iloc[tr],
            y[tr],
            eval_set=[(X_train_num.iloc[va], y[va])],
            verbose=False,
        )
        p = model.predict_proba(X_train_num.iloc[va])[:, 1]
        oof[va] = p
        used[va] = True

        m = metric_at_threshold(y[va], p, 0.5)
        trial.report(m["macro_f1"], step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    _, best = threshold_search(y[used], oof[used], THRESHOLD_GRID)
    return best["macro_f1"]


def optuna_objective_cat(trial):
    params = BASE_CAT.copy()
    params.update(
        {
            "iterations": trial.suggest_int("iterations", 1200, 3200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "depth": trial.suggest_int("depth", 5, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        }
    )

    oof = np.zeros(len(y), dtype=np.float64)
    used = np.zeros(len(y), dtype=bool)

    for fold, (tr, va) in enumerate(TUNE_SPLITS, 1):
        tr_pool = Pool(X_train_all.iloc[tr], y[tr], cat_features=CAT_COLS)
        va_pool = Pool(X_train_all.iloc[va], y[va], cat_features=CAT_COLS)

        model = CatBoostClassifier(**params)
        model.fit(tr_pool, eval_set=va_pool, verbose=False)

        p = model.predict_proba(va_pool)[:, 1]
        oof[va] = p
        used[va] = True

        m = metric_at_threshold(y[va], p, 0.5)
        trial.report(m["macro_f1"], step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    _, best = threshold_search(y[used], oof[used], THRESHOLD_GRID)
    return best["macro_f1"]


def optuna_objective_lgb(trial):
    params = BASE_LGB.copy()
    params.update(
        {
            "n_estimators": trial.suggest_int("n_estimators", 1000, 3200),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "max_depth": trial.suggest_int("max_depth", -1, 14),
            "min_child_samples": trial.suggest_int("min_child_samples", 30, 220),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", GLOBAL_SPW * 0.6, GLOBAL_SPW * 1.8
            ),
            "is_unbalance": trial.suggest_categorical("is_unbalance", [True, False]),
        }
    )

    oof = np.zeros(len(y), dtype=np.float64)
    used = np.zeros(len(y), dtype=bool)

    for fold, (tr, va) in enumerate(TUNE_SPLITS, 1):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_all.iloc[tr],
            y[tr],
            eval_set=[(X_train_all.iloc[va], y[va])],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(150, verbose=False)],
            categorical_feature=CAT_COLS,
        )

        p = model.predict_proba(X_train_all.iloc[va])[:, 1]
        oof[va] = p
        used[va] = True

        m = metric_at_threshold(y[va], p, 0.5)
        trial.report(m["macro_f1"], step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    _, best = threshold_search(y[used], oof[used], THRESHOLD_GRID)
    return best["macro_f1"]


def run_study(objective_fn, name):
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        pruner=MedianPruner(n_warmup_steps=1),
        study_name=name,
    )
    study.optimize(
        objective_fn,
        n_trials=OPTUNA_TRIALS_PER_MODEL,
        timeout=OPTUNA_TIMEOUT_SEC,
        show_progress_bar=False,
    )
    print(f"{name} best value:", study.best_value)
    print(f"{name} best params:", study.best_params)
    return study


tuned_params = {}

if ENABLE_XGB:
    study_xgb = run_study(optuna_objective_xgb, "xgb_v6")
    tuned_params["xgb"] = study_xgb.best_params

if ENABLE_CATBOOST:
    study_cat = run_study(optuna_objective_cat, "cat_v6")
    tuned_params["cat"] = study_cat.best_params

if ENABLE_LIGHTGBM:
    study_lgb = run_study(optuna_objective_lgb, "lgb_v6")
    tuned_params["lgb"] = study_lgb.best_params


In [ ]:
# -----------------------------
# CV training for OOF/test probs and feature importances
# -----------------------------
def get_primary_splits():
    if PRIMARY_CV_MODE == "time":
        return splits_time
    if PRIMARY_CV_MODE == "group":
        return splits_group
    return splits_old


PRIMARY_SPLITS = get_primary_splits()

fold_rows = []
feature_importance_rows = []
model_oof = {}
model_test = {}
fold_thresholds = {}


def train_xgb_branch():
    params = BASE_XGB.copy()
    params.update(tuned_params.get("xgb", {}))
    params.update(XGB_DEVICE_PARAMS)

    params_a = params.copy()
    params_b = params.copy()
    params_b["random_state"] = params_a.get("random_state", RANDOM_STATE) + 13
    params_b["max_depth"] = max(3, int(params_a["max_depth"]) - 1)
    params_b["min_child_weight"] = int(params_a["min_child_weight"]) + 2
    params_b["subsample"] = max(0.6, min(1.0, float(params_a["subsample"]) - 0.05))

    oof_a = np.zeros(len(y), dtype=np.float64)
    oof_b = np.zeros(len(y), dtype=np.float64)
    test_a = np.zeros(len(X_test_num), dtype=np.float64)
    test_b = np.zeros(len(X_test_num), dtype=np.float64)

    fold_thr = []

    for fold, (tr, va) in enumerate(PRIMARY_SPLITS, 1):
        t0 = time.time()

        m_a = XGBClassifier(**params_a)
        m_b = XGBClassifier(**params_b)

        m_a.fit(
            X_train_num.iloc[tr],
            y[tr],
            eval_set=[(X_train_num.iloc[va], y[va])],
            verbose=False,
        )
        m_b.fit(
            X_train_num.iloc[tr],
            y[tr],
            eval_set=[(X_train_num.iloc[va], y[va])],
            verbose=False,
        )

        p_va_a = m_a.predict_proba(X_train_num.iloc[va])[:, 1]
        p_va_b = m_b.predict_proba(X_train_num.iloc[va])[:, 1]
        p_va_blend = 0.5 * p_va_a + 0.5 * p_va_b

        oof_a[va] = p_va_a
        oof_b[va] = p_va_b

        test_a += m_a.predict_proba(X_test_num)[:, 1] / len(PRIMARY_SPLITS)
        test_b += m_b.predict_proba(X_test_num)[:, 1] / len(PRIMARY_SPLITS)

        _, best_fold = threshold_search(y[va], p_va_blend, THRESHOLD_GRID)
        fold_thr.append(float(best_fold["threshold"]))

        m05 = metric_at_threshold(y[va], p_va_blend, 0.5)
        fold_rows.append(
            {
                "model": "xgb_blend_internal",
                "fold": fold,
                "macro_f1_at_0_5": m05["macro_f1"],
                "fraud_f1_at_0_5": m05["fraud_f1"],
                "precision_at_0_5": m05["precision"],
                "recall_at_0_5": m05["recall"],
                "seconds": time.time() - t0,
                "cv_mode": PRIMARY_CV_MODE,
            }
        )

        imp_a = pd.Series(m_a.feature_importances_, index=NUMERIC_FEATURES)
        imp_b = pd.Series(m_b.feature_importances_, index=NUMERIC_FEATURES)
        imp = ((imp_a + imp_b) / 2.0).sort_values(ascending=False)
        for k, v in imp.head(40).items():
            feature_importance_rows.append(
                {"model": "xgb", "fold": fold, "feature": k, "importance": float(v)}
            )

        print(f"XGB fold {fold}/{len(PRIMARY_SPLITS)} done in {time.time() - t0:.1f}s")

    # optimize A/B blending weight on OOF
    blend_rows = []
    for w in np.arange(0.0, 1.0001, WEIGHT_GRID_STEP):
        p = w * oof_a + (1.0 - w) * oof_b
        m = metric_at_threshold(y, p, 0.5)
        blend_rows.append({"w_a": float(w), "macro_f1_at_0_5": m["macro_f1"]})
    blend_df = pd.DataFrame(blend_rows).sort_values("macro_f1_at_0_5", ascending=False)
    best_w = float(blend_df.iloc[0]["w_a"])

    oof = best_w * oof_a + (1.0 - best_w) * oof_b
    test = best_w * test_a + (1.0 - best_w) * test_b

    model_oof["xgb"] = oof
    model_test["xgb"] = test
    fold_thresholds["xgb"] = fold_thr

    print("Best internal XGB blend weight A:", best_w)


def train_cat_branch():
    params = BASE_CAT.copy()
    params.update(tuned_params.get("cat", {}))

    oof = np.zeros(len(y), dtype=np.float64)
    test = np.zeros(len(X_test_all), dtype=np.float64)
    fold_thr = []

    for fold, (tr, va) in enumerate(PRIMARY_SPLITS, 1):
        t0 = time.time()
        tr_pool = Pool(X_train_all.iloc[tr], y[tr], cat_features=CAT_COLS)
        va_pool = Pool(X_train_all.iloc[va], y[va], cat_features=CAT_COLS)
        te_pool = Pool(X_test_all, cat_features=CAT_COLS)

        model = CatBoostClassifier(**params)
        model.fit(tr_pool, eval_set=va_pool, verbose=False)

        p_va = model.predict_proba(va_pool)[:, 1]
        oof[va] = p_va
        test += model.predict_proba(te_pool)[:, 1] / len(PRIMARY_SPLITS)

        _, best_fold = threshold_search(y[va], p_va, THRESHOLD_GRID)
        fold_thr.append(float(best_fold["threshold"]))

        m05 = metric_at_threshold(y[va], p_va, 0.5)
        fold_rows.append(
            {
                "model": "catboost",
                "fold": fold,
                "macro_f1_at_0_5": m05["macro_f1"],
                "fraud_f1_at_0_5": m05["fraud_f1"],
                "precision_at_0_5": m05["precision"],
                "recall_at_0_5": m05["recall"],
                "seconds": time.time() - t0,
                "cv_mode": PRIMARY_CV_MODE,
            }
        )

        imp = pd.Series(
            model.get_feature_importance(tr_pool), index=X_train_all.columns
        ).sort_values(ascending=False)
        for k, v in imp.head(40).items():
            feature_importance_rows.append(
                {
                    "model": "catboost",
                    "fold": fold,
                    "feature": k,
                    "importance": float(v),
                }
            )

        print(
            f"CatBoost fold {fold}/{len(PRIMARY_SPLITS)} done in {time.time() - t0:.1f}s"
        )

    model_oof["cat"] = oof
    model_test["cat"] = test
    fold_thresholds["cat"] = fold_thr


def train_lgb_branch():
    params = BASE_LGB.copy()
    params.update(tuned_params.get("lgb", {}))

    oof = np.zeros(len(y), dtype=np.float64)
    test = np.zeros(len(X_test_all), dtype=np.float64)
    fold_thr = []

    for fold, (tr, va) in enumerate(PRIMARY_SPLITS, 1):
        t0 = time.time()
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train_all.iloc[tr],
            y[tr],
            eval_set=[(X_train_all.iloc[va], y[va])],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(150, verbose=False)],
            categorical_feature=CAT_COLS,
        )

        p_va = model.predict_proba(X_train_all.iloc[va])[:, 1]
        oof[va] = p_va
        test += model.predict_proba(X_test_all)[:, 1] / len(PRIMARY_SPLITS)

        _, best_fold = threshold_search(y[va], p_va, THRESHOLD_GRID)
        fold_thr.append(float(best_fold["threshold"]))

        m05 = metric_at_threshold(y[va], p_va, 0.5)
        fold_rows.append(
            {
                "model": "lightgbm",
                "fold": fold,
                "macro_f1_at_0_5": m05["macro_f1"],
                "fraud_f1_at_0_5": m05["fraud_f1"],
                "precision_at_0_5": m05["precision"],
                "recall_at_0_5": m05["recall"],
                "seconds": time.time() - t0,
                "cv_mode": PRIMARY_CV_MODE,
            }
        )

        imp = pd.Series(
            model.feature_importances_, index=X_train_all.columns
        ).sort_values(ascending=False)
        for k, v in imp.head(40).items():
            feature_importance_rows.append(
                {
                    "model": "lightgbm",
                    "fold": fold,
                    "feature": k,
                    "importance": float(v),
                }
            )

        print(
            f"LightGBM fold {fold}/{len(PRIMARY_SPLITS)} done in {time.time() - t0:.1f}s"
        )

    model_oof["lgb"] = oof
    model_test["lgb"] = test
    fold_thresholds["lgb"] = fold_thr


if ENABLE_XGB:
    train_xgb_branch()
if ENABLE_CATBOOST:
    train_cat_branch()
if ENABLE_LIGHTGBM:
    train_lgb_branch()

fold_scores_df = pd.DataFrame(fold_rows)
display(
    fold_scores_df.groupby("model", as_index=False)[
        ["macro_f1_at_0_5", "fraud_f1_at_0_5"]
    ]
    .mean()
    .sort_values("macro_f1_at_0_5", ascending=False)
)


In [ ]:
# -----------------------------
# Strict-vs-old CV comparison (side-by-side)
# -----------------------------
def evaluate_cv_mode(model_name, splits, params):
    rows = []
    for fold, (tr, va) in enumerate(splits, 1):
        if model_name == "xgb":
            m = XGBClassifier(**params)
            m.fit(
                X_train_num.iloc[tr],
                y[tr],
                eval_set=[(X_train_num.iloc[va], y[va])],
                verbose=False,
            )
            p = m.predict_proba(X_train_num.iloc[va])[:, 1]
        elif model_name == "cat":
            m = CatBoostClassifier(**params)
            tr_pool = Pool(X_train_all.iloc[tr], y[tr], cat_features=CAT_COLS)
            va_pool = Pool(X_train_all.iloc[va], y[va], cat_features=CAT_COLS)
            m.fit(tr_pool, eval_set=va_pool, verbose=False)
            p = m.predict_proba(va_pool)[:, 1]
        else:
            m = lgb.LGBMClassifier(**params)
            m.fit(
                X_train_all.iloc[tr],
                y[tr],
                eval_set=[(X_train_all.iloc[va], y[va])],
                eval_metric="binary_logloss",
                callbacks=[lgb.early_stopping(120, verbose=False)],
                categorical_feature=CAT_COLS,
            )
            p = m.predict_proba(X_train_all.iloc[va])[:, 1]

        _, best = threshold_search(y[va], p, THRESHOLD_GRID)
        rows.append({"fold": fold, **best})

    df = pd.DataFrame(rows)
    return {
        "macro_f1": float(df["macro_f1"].mean()),
        "fraud_f1": float(df["fraud_f1"].mean()),
        "precision": float(df["precision"].mean()),
        "recall": float(df["recall"].mean()),
    }


cv_compare_rows = []

xgb_compare_params = BASE_XGB.copy()
xgb_compare_params.update(tuned_params.get("xgb", {}))
xgb_compare_params.update(XGB_DEVICE_PARAMS)

old_score = evaluate_cv_mode("xgb", splits_old, xgb_compare_params)
time_score = evaluate_cv_mode("xgb", splits_time, xgb_compare_params)
group_score = evaluate_cv_mode("xgb", splits_group, xgb_compare_params)

for mode_name, m in [
    ("old_stratified", old_score),
    ("time_based", time_score),
    ("grouped", group_score),
]:
    cv_compare_rows.append({"model": "xgb_single", "cv_mode": mode_name, **m})

cv_comparison_df = pd.DataFrame(cv_compare_rows).sort_values(
    ["model", "macro_f1"], ascending=[True, False]
)
display(cv_comparison_df)


In [ ]:
# -----------------------------
# Ensemble optimization + latest-time holdout selection
# -----------------------------
def build_combo_prob(weights, keys, pred_map):
    p = np.zeros(len(next(iter(pred_map.values()))), dtype=np.float64)
    for k, w in zip(keys, weights):
        p += w * pred_map[k]
    return p


def optimize_weights(keys, oof_map):
    if len(keys) == 1:
        return [1.0]

    best = None

    if len(keys) == 2:
        for w0 in np.arange(0.0, 1.0001, WEIGHT_GRID_STEP):
            ws = [float(w0), float(1.0 - w0)]
            p = build_combo_prob(ws, keys, oof_map)
            m = metric_at_threshold(y, p, 0.5)
            row = {"weights": ws, "macro_f1_at_0_5": m["macro_f1"]}
            if best is None or row["macro_f1_at_0_5"] > best["macro_f1_at_0_5"]:
                best = row
        return best["weights"]

    if len(keys) == 3:
        for w0 in np.arange(0.0, 1.0001, WEIGHT_GRID_STEP):
            for w1 in np.arange(0.0, 1.0001 - w0, WEIGHT_GRID_STEP):
                w2 = 1.0 - w0 - w1
                if w2 < -1e-12:
                    continue
                ws = [float(w0), float(w1), float(max(0.0, w2))]
                s = sum(ws)
                ws = [w / s for w in ws]
                p = build_combo_prob(ws, keys, oof_map)
                m = metric_at_threshold(y, p, 0.5)
                row = {"weights": ws, "macro_f1_at_0_5": m["macro_f1"]}
                if best is None or row["macro_f1_at_0_5"] > best["macro_f1_at_0_5"]:
                    best = row
        return best["weights"]

    raise ValueError("Unsupported combo size")


def model_predict_holdout_and_test(model_key):
    if model_key == "xgb":
        params = BASE_XGB.copy()
        params.update(tuned_params.get("xgb", {}))
        params.update(XGB_DEVICE_PARAMS)

        a = XGBClassifier(**params)
        b_params = params.copy()
        b_params["random_state"] = params.get("random_state", RANDOM_STATE) + 13
        b_params["max_depth"] = max(3, int(b_params["max_depth"]) - 1)
        b_params["min_child_weight"] = int(b_params["min_child_weight"]) + 2
        b = XGBClassifier(**b_params)

        a.fit(
            X_train_num.iloc[latest_tr],
            y[latest_tr],
            eval_set=[(X_train_num.iloc[latest_va], y[latest_va])],
            verbose=False,
        )
        b.fit(
            X_train_num.iloc[latest_tr],
            y[latest_tr],
            eval_set=[(X_train_num.iloc[latest_va], y[latest_va])],
            verbose=False,
        )

        p_hold = (
            0.5 * a.predict_proba(X_train_num.iloc[latest_va])[:, 1]
            + 0.5 * b.predict_proba(X_train_num.iloc[latest_va])[:, 1]
        )
        return p_hold

    if model_key == "cat":
        params = BASE_CAT.copy()
        params.update(tuned_params.get("cat", {}))

        tr_pool = Pool(X_train_all.iloc[latest_tr], y[latest_tr], cat_features=CAT_COLS)
        va_pool = Pool(X_train_all.iloc[latest_va], y[latest_va], cat_features=CAT_COLS)
        m = CatBoostClassifier(**params)
        m.fit(tr_pool, eval_set=va_pool, verbose=False)
        return m.predict_proba(va_pool)[:, 1]

    if model_key == "lgb":
        params = BASE_LGB.copy()
        params.update(tuned_params.get("lgb", {}))
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_train_all.iloc[latest_tr],
            y[latest_tr],
            eval_set=[(X_train_all.iloc[latest_va], y[latest_va])],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(120, verbose=False)],
            categorical_feature=CAT_COLS,
        )
        return m.predict_proba(X_train_all.iloc[latest_va])[:, 1]

    raise ValueError(model_key)


available_models = [k for k in ["xgb", "cat", "lgb"] if k in model_oof]
required_combos = []
if "xgb" in available_models:
    required_combos.append(("xgb_only", ["xgb"]))
if all(k in available_models for k in ["xgb", "cat"]):
    required_combos.append(("xgb_cat", ["xgb", "cat"]))
if all(k in available_models for k in ["xgb", "lgb"]):
    required_combos.append(("xgb_lgb", ["xgb", "lgb"]))
if all(k in available_models for k in ["xgb", "cat", "lgb"]):
    required_combos.append(("xgb_cat_lgb", ["xgb", "cat", "lgb"]))

holdout_pred_map = {k: model_predict_holdout_and_test(k) for k in available_models}

ensemble_rows = []
final_test_candidates = {}

for name, keys in required_combos:
    weights = optimize_weights(keys, model_oof)
    oof_prob = build_combo_prob(weights, keys, model_oof)
    test_prob = build_combo_prob(weights, keys, model_test)
    holdout_prob = build_combo_prob(weights, keys, holdout_pred_map)

    _, best_global = threshold_search(y, oof_prob, THRESHOLD_GRID)
    global_thr = float(best_global["threshold"])

    per_model_fold_thrs = []
    for k in keys:
        if k in fold_thresholds and len(fold_thresholds[k]) > 0:
            per_model_fold_thrs.extend(fold_thresholds[k])
    fold_avg_thr = (
        float(np.mean(per_model_fold_thrs)) if per_model_fold_thrs else global_thr
    )

    hold_global = metric_at_threshold(y[latest_va], holdout_prob, global_thr)
    hold_foldavg = metric_at_threshold(y[latest_va], holdout_prob, fold_avg_thr)

    if hold_global["macro_f1"] >= hold_foldavg["macro_f1"]:
        sel_thr = global_thr
        thr_mode = "global"
        sel_hold = hold_global
    else:
        sel_thr = fold_avg_thr
        thr_mode = "fold_avg"
        sel_hold = hold_foldavg

    extreme_penalty = abs(sel_thr - 0.5)

    row = {
        "ensemble": name,
        "models": "+".join(keys),
        "weights": str([round(w, 4) for w in weights]),
        "oof_macro_f1_best_global": float(best_global["macro_f1"]),
        "oof_best_global_threshold": global_thr,
        "oof_fold_avg_threshold": fold_avg_thr,
        "selected_threshold": sel_thr,
        "selected_threshold_mode": thr_mode,
        "latest_holdout_macro_f1": float(sel_hold["macro_f1"]),
        "latest_holdout_fraud_f1": float(sel_hold["fraud_f1"]),
        "latest_holdout_precision": float(sel_hold["precision"]),
        "latest_holdout_recall": float(sel_hold["recall"]),
        "threshold_extreme_penalty": float(extreme_penalty),
    }
    ensemble_rows.append(row)

    final_test_candidates[name] = (test_prob, sel_thr)

ensemble_df = pd.DataFrame(ensemble_rows)
ensemble_df = ensemble_df.sort_values(
    ["latest_holdout_macro_f1", "latest_holdout_recall", "threshold_extreme_penalty"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(ensemble_df)

best_ensemble_name = ensemble_df.iloc[0]["ensemble"]
best_test_prob, best_thr = final_test_candidates[best_ensemble_name]
final_pred = (best_test_prob >= best_thr).astype(int)

submission = pd.DataFrame({ID_COL: test_df[ID_COL], LABEL_COL: final_pred})
assert len(submission) == len(test_df), "Submission length mismatch"
assert submission.columns.tolist() == [ID_COL, LABEL_COL], "Submission schema mismatch"
assert set(submission[LABEL_COL].unique()).issubset({0, 1}), (
    "Submission target must be binary"
)


In [ ]:
# -----------------------------
# Save artifacts, report, and optional submit
# -----------------------------
feature_importance_df = pd.DataFrame(feature_importance_rows)
if not feature_importance_df.empty:
    feature_importance_df = (
        feature_importance_df.groupby(["model", "feature"], as_index=False)[
            "importance"
        ]
        .mean()
        .sort_values(["model", "importance"], ascending=[True, False])
    )

fold_scores_df.to_csv(FOLD_TABLE_PATH, index=False)
ensemble_df.to_csv(ENSEMBLE_TABLE_PATH, index=False)
cv_comparison_df.to_csv(CV_COMPARISON_PATH, index=False)
if not feature_importance_df.empty:
    feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

submission.to_csv(OUTPUT_PATH, index=False)

best_row = ensemble_df.iloc[0].to_dict()
diagnostics = pd.DataFrame(
    [
        {
            "notebook_slug": NOTEBOOK_SLUG,
            "primary_cv_mode": PRIMARY_CV_MODE,
            "n_rows_train": int(len(X_train_all)),
            "n_rows_test": int(len(X_test_all)),
            "n_features": int(X_train_all.shape[1]),
            "best_ensemble": best_row["ensemble"],
            "best_models": best_row["models"],
            "best_weights": best_row["weights"],
            "selected_threshold": float(best_row["selected_threshold"]),
            "selected_threshold_mode": best_row["selected_threshold_mode"],
            "latest_holdout_macro_f1": float(best_row["latest_holdout_macro_f1"]),
            "latest_holdout_fraud_f1": float(best_row["latest_holdout_fraud_f1"]),
            "predicted_fraud_rows_test": int(submission[LABEL_COL].sum()),
            "predicted_fraud_rate_test": float(submission[LABEL_COL].mean()),
            "run_submission": RUN_SUBMISSION,
        }
    ]
)
diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)

display(diagnostics)
display(submission.head())
print("Saved submission:", OUTPUT_PATH)
print("Saved diagnostics:", DIAGNOSTICS_PATH)
print("Saved fold table:", FOLD_TABLE_PATH)
print("Saved ensemble table:", ENSEMBLE_TABLE_PATH)
print("Saved CV comparison:", CV_COMPARISON_PATH)
if not feature_importance_df.empty:
    print("Saved feature importance:", FEATURE_IMPORTANCE_PATH)

if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")
    msg = (
        f"{NOTEBOOK_SLUG} | holdout_macro_f1={best_row['latest_holdout_macro_f1']:.6f} "
        f"ensemble={best_row['ensemble']} thr={best_row['selected_threshold']:.4f}"
    )
    api.competition_submit(
        file_name=str(OUTPUT_PATH), message=msg, competition=COMPETITION, quiet=False
    )
    print("Submission sent with message:", msg)
else:
    print("RUN_SUBMISSION=False, skipped auto-submit")


## Why These Changes

- **Stricter validation suite**: adds time-based, grouped, and latest-time holdout views to reduce optimistic CV and better match leaderboard drift.
- **Past-only historical features**: rolling account/pair behavior helps detect rare fraud patterns while staying leakage-safe.
- **Three model families**: XGB + CatBoost + LightGBM improves diversity and robustness under distribution shift.
- **Optuna with pruning**: improves parameter quality with realistic runtime budget.
- **Ensemble + threshold strategy**: optimizes model mix and decision threshold directly for Macro F1, including global vs fold-averaged threshold.


## First Submission Recommendation

Submit the top-ranked row from `ensemble_table_<NOTEBOOK_SLUG>.csv` (selected by latest-time holdout Macro F1 with recall-aware tie-break).
In most runs this is expected to be **`xgb_cat_lgb`** with a tuned threshold rather than fixed `0.5`.
